# Virginia — Title 38.2 (Insurance) → `data/virginia/ins_codes/*.md`

Virginia’s **insurance** statutes are **Title 38.2 — Insurance** of the **Code of Virginia**. On **Justia**, the crawl root is **[`/codes/virginia/title-38-2/`](https://law.justia.com/codes/virginia/title-38-2/)**; sections live under **`…/title-38-2/chapter-…/section-…/`** (same structural idea as **`south_carolina.ipynb`**).

**Cloudflare** often blocks plain **`httpx`**; this notebook uses **`curl_cffi`** with **`impersonate="chrome120"`**.

**Discovery:** BFS from the Title 38.2 index, following only paths under **`/codes/virginia/title-38-2/`** that are **not** section pages, skipping **`appendix`**, **`chronological-history`**, and **`title-notes`**; collect every section link (~**1,893** sections across ~**69** chapter index pages).

**Download:** text from **`div.primary-content`**, with Justia boilerplate stripped. Files are **`VA_sec_<slug>.md`** where **`<slug>`** normalizes the segment after **`section-`** (e.g. `38-2-100` → `VA_sec_38_2_100.md`). Display maps the Justia slug prefix **`38-2-`** to **`38.2-`** (e.g. **`Va. Code § 38.2-100`**).

Config: **MAX_SECTIONS** (**0** = all), **MAX_DISCOVERY_PAGES** (**0** = no cap). **REUSE_DISCOVERED_URLS** skips discovery when **`_va_title38_2_section_urls.txt`** exists.

Run with the **`ins_ipynb/`** directory as cwd. Then **`python -m app.ingest`** from the project root.


In [1]:
%pip install -q curl_cffi beautifulsoup4


You should consider upgrading via the '/Users/apps/Downloads/ZProjects/RAG/.venv/bin/python -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.


In [2]:
from __future__ import annotations

import re
import time
from pathlib import Path
from urllib.parse import urljoin, urlparse

from bs4 import BeautifulSoup
from curl_cffi import requests as curl_requests

BASE = "https://law.justia.com"
PATH_PREFIX = "/codes/virginia/title-38-2"
TITLE_INDEX = f"{BASE}{PATH_PREFIX}/"

OUT_DIR = Path("data") / "virginia" / "ins_codes"
OUT_DIR.mkdir(parents=True, exist_ok=True)

CURL_IMPERSONATE = "chrome120"
REQUEST_DELAY_SEC = 0.12
TIMEOUT = 60.0

MAX_SECTIONS = 0
MAX_DISCOVERY_PAGES = 0

SKIP_EXISTING = True

DISCOVERED_LIST = OUT_DIR / "_va_title38_2_section_urls.txt"
REUSE_DISCOVERED_URLS = True

SKIP_PATH_SUBSTR = ("appendix", "chronological-history", "title-notes")


In [3]:
def curl_get(url: str) -> str:
    time.sleep(REQUEST_DELAY_SEC)
    r = curl_requests.get(url, impersonate=CURL_IMPERSONATE, timeout=TIMEOUT)
    r.raise_for_status()
    return r.text


def path_key(u: str) -> str:
    return urlparse(u).path.rstrip("/")


def skip_path(p: str) -> bool:
    low = p.lower()
    return any(s in low for s in SKIP_PATH_SUBSTR)


def discover_section_urls() -> list[str]:
    """BFS title-38-2 index + chapter pages; collect section URLs."""
    from collections import deque

    start = TITLE_INDEX
    seen: set[str] = set()
    in_q: set[str] = {path_key(start).lower()}
    q: deque[str] = deque([start])
    sections: set[str] = set()
    fetches = 0
    while q:
        if MAX_DISCOVERY_PAGES and fetches >= MAX_DISCOVERY_PAGES:
            break
        url = q.popleft()
        pk = path_key(url).lower()
        in_q.discard(pk)
        if pk in seen:
            continue
        if "/section-" in pk.lower():
            continue
        seen.add(pk)
        html = curl_get(url)
        fetches += 1
        if fetches % 20 == 0:
            print(f"… discovery fetch {fetches}, queue={len(q)}, sections={len(sections)}")
        soup = BeautifulSoup(html, "html.parser")
        for a in soup.find_all("a", href=True):
            absu = urljoin(url, a["href"])
            p = path_key(absu).lower()
            if not p.startswith(PATH_PREFIX):
                continue
            if skip_path(p):
                continue
            if "/section-" in p.lower():
                sections.add(BASE + p + "/")
            else:
                if p in seen or p in in_q:
                    continue
                in_q.add(p)
                q.append(BASE + p + "/")
    return sorted(sections, key=lambda u: label_sort_key(section_label_from_url(u)))


def section_label_from_url(url: str) -> str:
    path = path_key(url)
    low = path.lower()
    if "/section-" not in low:
        raise ValueError(f"not a section URL: {url!r}")
    return path.rsplit("/section-", 1)[1]


def label_sort_key(label: str) -> tuple:
    out: list[tuple[int, int | str]] = []
    for part in label.split("-"):
        if part.isdigit():
            out.append((0, int(part)))
        else:
            out.append((1, part.lower()))
    return tuple(out)


def label_to_display_citation(label: str) -> str:
    """Justia slug 38-2-100 → Va. Code § 38.2-100"""
    low = label.lower()
    if low.startswith("38-2-"):
        return "Va. Code § 38.2-" + label[5:]
    return f"Va. Code § {label}"


def label_to_filename(label: str) -> str:
    safe = re.sub(r"[^0-9a-zA-Z]+", "_", label).strip("_").lower()
    return f"VA_sec_{safe}.md"


def extract_primary_text(html: str) -> tuple[str, str]:
    soup = BeautifulSoup(html, "html.parser")
    title_el = soup.find("title")
    title_txt = title_el.get_text(strip=True) if title_el else ""
    pc = soup.select_one("div.primary-content")
    if pc:
        text = pc.get_text("\n", strip=True)
    else:
        main = soup.find("main") or soup.find("article")
        text = main.get_text("\n", strip=True) if main else soup.get_text("\n", strip=True)
    return title_txt, text


def strip_justia_boilerplate(text: str) -> str:
    drop_prefixes = (
        "Go to Previous Versions",
        "View All Versions",
        "Learn more",
        "This media-neutral citation",
    )
    lines = text.split("\n")
    out: list[str] = []
    skip_until_substantive = True
    for line in lines:
        s = line.strip()
        if not s:
            if not skip_until_substantive:
                out.append("")
            continue
        if any(s.startswith(p) for p in drop_prefixes):
            continue
        if s.startswith("20") and ("Code of Virginia" in s or "Va. Code" in s):
            continue
        if s in {"Next", "Previous", "Universal Citation:"}:
            continue
        if s.startswith("Code of Virginia"):
            continue
        skip_until_substantive = False
        out.append(s)
    return "\n".join(out).strip()


def download_title_38_2() -> dict[str, int]:
    if REUSE_DISCOVERED_URLS and DISCOVERED_LIST.exists() and DISCOVERED_LIST.stat().st_size > 50:
        raw = [ln.strip() for ln in DISCOVERED_LIST.read_text(encoding="utf-8").splitlines() if ln.strip()]
        all_urls = sorted(raw, key=lambda u: label_sort_key(section_label_from_url(u)))
        print(f"Loaded {len(all_urls)} section URLs from {DISCOVERED_LIST.name} (skipped discovery)")
    else:
        found = discover_section_urls()
        print(f"Discovered {len(found)} section URLs under Title 38.2")
        all_urls = sorted(found, key=lambda u: label_sort_key(section_label_from_url(u)))
        DISCOVERED_LIST.write_text("\n".join(all_urls) + "\n", encoding="utf-8")

    todo = all_urls if not MAX_SECTIONS else all_urls[:MAX_SECTIONS]
    if MAX_SECTIONS:
        print(f"Limited downloads to first {len(todo)} sections (MAX_SECTIONS)")

    wrote = skipped = failed = 0
    for i, sec_url in enumerate(todo, 1):
        label = section_label_from_url(sec_url)
        disp = label_to_display_citation(label)
        dest = OUT_DIR / label_to_filename(label)
        if SKIP_EXISTING and dest.exists() and dest.stat().st_size > 80:
            skipped += 1
        else:
            try:
                html = curl_get(sec_url)
                head_t, body_t = extract_primary_text(html)
                body_t = strip_justia_boilerplate(body_t)
                title = head_t or f"Code of Virginia {disp}"
                md = (
                    f"# {title}\n\n"
                    f"**Code of Virginia — Title 38.2 (Insurance)**\n\n"
                    f"**Source (Justia mirror):** {sec_url}\n\n"
                    f"**Verify on official site:** [Virginia Law — Title 38.2](https://law.lis.virginia.gov/vacodefull/title38.2/)\n\n"
                    f"**Section (URL slug):** {label}\n\n"
                    f"**Citation (display):** {disp}\n\n"
                    f"---\n\n"
                    f"{body_t}\n"
                )
                dest.write_text(md, encoding="utf-8")
                wrote += 1
            except Exception as e:
                print(f"FAIL {label}: {e}")
                failed += 1
        if i % 200 == 0:
            print(f"… {i}/{len(todo)} (wrote={wrote} skipped={skipped} failed={failed})")

    print(f"Done. wrote={wrote} skipped={skipped} failed={failed} → {OUT_DIR.resolve()}")
    return {"wrote": wrote, "skipped": skipped, "failed": failed}


download_title_38_2()


… discovery fetch 20, queue=49, sections=720
… discovery fetch 40, queue=29, sections=1400
… discovery fetch 60, queue=9, sections=1808
Discovered 1893 section URLs under Title 38.2
… 200/1893 (wrote=200 skipped=0 failed=0)
… 400/1893 (wrote=400 skipped=0 failed=0)
… 600/1893 (wrote=600 skipped=0 failed=0)
… 800/1893 (wrote=800 skipped=0 failed=0)
… 1000/1893 (wrote=1000 skipped=0 failed=0)
… 1200/1893 (wrote=1200 skipped=0 failed=0)
… 1400/1893 (wrote=1400 skipped=0 failed=0)
… 1600/1893 (wrote=1600 skipped=0 failed=0)
… 1800/1893 (wrote=1800 skipped=0 failed=0)
Done. wrote=1893 skipped=0 failed=0 → /Users/apps/Downloads/ZProjects/RAG/ins_ipynb/data/virginia/ins_codes


{'wrote': 1893, 'skipped': 0, 'failed': 0}

## Next step

`python -m app.ingest` from the project root.
